# Phase 2 — Synthetic customer assignment

The Kaggle dataset has no persistent customer identifier. This notebook validates a deterministic synthetic identity layer; it does **not** claim to reconstruct real people.

Stable profiles are stratified by sender age group, state, and bank. Activity, device/network preferences, and behavioral archetypes are controlled synthetic assumptions. The `synthetic_archetype` field is validation metadata and must not be used as a future model feature.

In [ ]:
from pathlib import Path
import json
import pandas as pd


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        path = candidate / "ml" / "data" / "processed" / "customers.csv"
        if path.exists():
            return candidate
    raise FileNotFoundError("Run `python -m ml.src.assign_customers` first")


ROOT = find_repo_root()
PROCESSED = ROOT / "ml" / "data" / "processed"
customers = pd.read_csv(PROCESSED / "customers.csv", parse_dates=["first_transaction_at", "last_transaction_at"])
transactions = pd.read_csv(PROCESSED / "transactions_with_customers.csv", parse_dates=["timestamp"])
summary = json.loads((PROCESSED / "customer_assignment_summary.json").read_text())

print("Customers:", customers.shape)
print("Transactions:", transactions.shape)
summary

In [ ]:
display(customers.head(3))
display(transactions.head(3))

print("Customer columns:", customers.columns.tolist())
print("Transaction columns:", transactions.columns.tolist())

In [ ]:
counts = transactions.groupby("customer_id").size()
print(counts.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

counts.plot.hist(bins=40, title="Transactions per synthetic customer", xlabel="Transactions")

In [ ]:
assert len(customers) == 10_000
assert len(transactions) == 250_000
assert transactions["customer_id"].notna().all()
assert transactions["transaction_id"].is_unique
assert transactions["customer_id"].nunique() == len(customers)
assert counts.min() >= 5

print("Every transaction has exactly one customer and every customer has at least five transactions.")

In [ ]:
profiles = customers.set_index("customer_id")
stable_checks = {
    "age_group": transactions["sender_age_group"].eq(transactions["customer_id"].map(profiles["age_group"])),
    "state": transactions["sender_state"].eq(transactions["customer_id"].map(profiles["state"])),
    "bank": transactions["sender_bank"].eq(transactions["customer_id"].map(profiles["bank"])),
}

for field, valid in stable_checks.items():
    print(field, "mismatches:", int((~valid).sum()))

In [ ]:
device_consistency = transactions["device_type"].eq(
    transactions["customer_id"].map(profiles["primary_device"])
).mean()
network_consistency = transactions["network_type"].eq(
    transactions["customer_id"].map(profiles["primary_network"])
).mean()

print(f"Primary-device consistency: {device_consistency:.2%}")
print(f"Primary-network consistency: {network_consistency:.2%}")

In [ ]:
analysis = transactions.merge(
    customers[["customer_id", "synthetic_archetype"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)
analysis["failed"] = analysis["transaction_status"].eq("FAILED")
analysis["night"] = analysis["hour_of_day"].ge(20) | analysis["hour_of_day"].le(5)

archetype_profile = analysis.groupby("synthetic_archetype").agg(
    customers=("customer_id", "nunique"),
    transactions=("transaction_id", "size"),
    mean_amount=("amount_inr", "mean"),
    failure_rate=("failed", "mean"),
    weekend_share=("is_weekend", "mean"),
    night_share=("night", "mean"),
)
archetype_profile

In [ ]:
chronological = transactions.groupby("customer_id")["timestamp"].apply(
    lambda values: values.is_monotonic_increasing
)
assert chronological.all()

first_and_last = transactions.groupby("customer_id")["timestamp"].agg(["min", "max"])
assert first_and_last["min"].eq(profiles["first_transaction_at"]).all()
assert first_and_last["max"].eq(profiles["last_transaction_at"]).all()
print("All customer histories are chronological and profile date bounds are valid.")

## Interpretation and limits

- Sender age group, state, and bank are perfectly stable by construction.
- Device and network are preferences, not immutable identity fields, so controlled variation is expected.
- Archetypes intentionally shape assignment and are synthetic assumptions—not observed Kaggle labels.
- No recovery outcome has been generated and no model has been trained.
- Phase 3 must compute every historical feature using only rows strictly earlier than the transaction being scored.